# Twitter-XLM-RoBERTa-base: eksperimen E0-E7 per bahasa

## 1. Library dan konfigurasi

In [7]:
import os

# Tetapkan backend sebelum datasets dan transformers diimpor.
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import re
import gc
import json
import hashlib
import shutil
import unicodedata
import io
from zipfile import ZipFile, ZIP_DEFLATED
from pathlib import Path
from itertools import combinations
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import emoji
from IPython.display import display, FileLink
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments, set_seed,
)

# Delapan skenario untuk setiap bahasa secara terpisah.
LANGUAGES = ["en", "it", "nl"]
MODEL_NAMES = ["cardiffnlp/twitter-xlm-roberta-base"]
# Untuk satu bahasa saja: LANGUAGES = ["en"]
# Semua model: MODEL_NAMES = ["cardiffnlp/twitter-xlm-roberta-base", "FacebookAI/xlm-roberta-base",
#                             "google-bert/bert-base-multilingual-uncased"]

RUN_NAMES = [
    "E0_legacy", "E1_keep_emoji", "E2_keep_case", "E3_no_weights",
    "E4_title_pair", "E5_title_desc_pair", "E6_lr_warmup", "E7_focal_loss",
]
SEED = 42
MAX_LENGTH = 384  # Sama dengan notebook lama
NUM_EPOCHS = 10
PATIENCE = 3
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 16
GRADIENT_CHECKPOINTING = True
EVAL_ACCUMULATION_STEPS = 1
WEIGHT_DECAY = 0.01
USE_DRIVE = False  # True: mount Drive untuk menyimpan hasil permanen
SKIP_FINISHED = True  # Aman melanjutkan setelah runtime terputus
RUN_FINAL_TEST = False  # Tetap False selama tuning E0-E7

IN_COLAB = "google.colab" in sys.modules
candidates = [Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd() / "Analisis" / "data"]
DATA_DIR = next(
    (path for path in candidates if any((path / f"{lang}-splitting").is_dir() for lang in LANGUAGES)),
    Path("/content/data") if IN_COLAB else Path.cwd() / "data",
).resolve()
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/StereoQueerEval_E0_E6")
else:
    OUT_DIR = (
        Path("/content/ablation_results")
        if IN_COLAB
        else DATA_DIR.parent / "results" / "ablation_multibahasa_e0_e6"
    ).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODELS_FILE = OUT_DIR.parent / "best_models.json"
best_models = json.loads(BEST_MODELS_FILE.read_text(encoding="utf-8")) if BEST_MODELS_FILE.exists() else {}

LABELS = ["no", "yes_implicit", "yes_explicit"]
LABEL2ID = {label: idx for idx, label in enumerate(LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}
assert set(LANGUAGES).issubset({"en", "it", "nl"}) and LANGUAGES
assert MODEL_NAMES
print("Eksperimen:", RUN_NAMES)
print("Bahasa:", LANGUAGES)
print("Model:", MODEL_NAMES)
print("Total pelatihan:", len(LANGUAGES) * len(MODEL_NAMES) * len(RUN_NAMES))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK TERDETEKSI")
print("Folder hasil:", OUT_DIR)


def cleanup():
    """Lepaskan objek tidak terpakai dan cache GPU antar-run."""
    plt.close("all")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Eksperimen: ['E0_legacy', 'E1_keep_emoji', 'E2_keep_case', 'E3_no_weights', 'E4_title_pair', 'E5_title_desc_pair', 'E6_lr_warmup', 'E7_focal_loss']
Bahasa: ['en', 'it', 'nl']
Model: ['cardiffnlp/twitter-xlm-roberta-base']
Total pelatihan: 24
GPU: NVIDIA GeForce RTX 3060 Ti
Folder hasil: E:\Projek\PTM\TBP\Analisis\results\ablation_multibahasa_e0_e6


## 2. Lokasi data dan upload opsional

In [8]:
DATA_DIR.mkdir(parents=True, exist_ok=True)


def data_path(lang, split):
    name = f"{lang}-{split}.csv"
    nested = DATA_DIR / f"{lang}-splitting" / name
    return nested if nested.is_file() else DATA_DIR / name


needed_names = [f"{lang}-{split}.csv" for lang in LANGUAGES for split in ("train", "val", "test")]
missing = [
    f"{lang}-{split}.csv"
    for lang in LANGUAGES
    for split in ("train", "val", "test")
    if not data_path(lang, split).is_file()
]
if missing:
    if not IN_COLAB:
        raise FileNotFoundError(f"CSV belum ditemukan di {DATA_DIR}: {missing}")
    from google.colab import files

    print("Unggah CSV atau ZIP untuk bahasa terpilih:", missing)
    uploaded = files.upload()
    for filename, blob in uploaded.items():
        if filename.lower().endswith(".zip"):
            with ZipFile(io.BytesIO(blob)) as z:
                for entry in z.infolist():
                    base = Path(entry.filename).name
                    if not entry.is_dir() and base in needed_names:
                        (DATA_DIR / base).write_bytes(z.read(entry))
        elif filename in needed_names:
            (DATA_DIR / filename).write_bytes(blob)
    if uploaded:
        del blob
    del uploaded
    cleanup()
missing = [
    str(data_path(lang, split))
    for lang in LANGUAGES
    for split in ("train", "val", "test")
    if not data_path(lang, split).is_file()
]
if missing:
    raise FileNotFoundError("CSV belum lengkap: " + ", ".join(missing))
print("Semua split bahasa terpilih tersedia. Tidak ada splitting ulang.")


Semua split bahasa terpilih tersedia. Tidak ada splitting ulang.


## 3. Baca data dan periksa overlap

In [9]:
DATA = {}
for lang in LANGUAGES:
    frames = {}
    for split in ("train", "val", "test"):
        path = data_path(lang, split)
        frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        required = {"StereoQueerEval_id", "yt_comment", "yt_title", "yt_description", "hate_speech"}
        assert required <= set(frame.columns), f"Kolom tidak lengkap: {path}"
        assert not frame.empty and frame["StereoQueerEval_id"].is_unique
        assert frame["StereoQueerEval_id"].str.strip().ne("").all()
        assert frame["yt_comment"].str.strip().ne("").all()
        assert set(frame["hate_speech"]) == set(LABELS)
        frames[split] = frame

    for left, right in combinations(frames, 2):
        for column in ("StereoQueerEval_id", "yt_comment"):
            left_values = frames[left][column].map(
                lambda text: re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text).casefold()).strip()
            )
            right_values = frames[right][column].map(
                lambda text: re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text).casefold()).strip()
            )
            assert not set(left_values) & set(right_values), f"Overlap {column}: {lang} {left}/{right}"
    DATA[lang] = frames
    print(lang.upper(), {split: len(frame) for split, frame in frames.items()})


EN {'train': 2111, 'val': 292, 'test': 586}
IT {'train': 1699, 'val': 220, 'test': 481}
NL {'train': 1556, 'val': 226, 'test': 456}


## 4. Konfigurasi E0-E7

In [10]:
EXPERIMENTS = {
    "E0_legacy": {
        "preproc": "legacy", "context": "none", "weighted": True,
        "lr": 1e-05, "warmup": 0.0,
    },
    "E1_keep_emoji": {
        "preproc": "keep_emoji", "context": "none", "weighted": True,
        "lr": 1e-05, "warmup": 0.0,
    },
    "E2_keep_case": {
        "preproc": "raw", "context": "none", "weighted": True,
        "lr": 1e-05, "warmup": 0.0,
    },
    "E3_no_weights": {
        "preproc": "raw", "context": "none", "weighted": False,
        "lr": 1e-05, "warmup": 0.0,
    },
    "E4_title_pair": {
        "preproc": "raw", "context": "title", "weighted": True,
        "lr": 1e-05, "warmup": 0.0,
    },
    "E5_title_desc_pair": {
        "preproc": "raw", "context": "title_desc", "weighted": True,
        "lr": 1e-05, "warmup": 0.0,
    },
    "E6_lr_warmup": {
        "preproc": "raw", "context": "title", "weighted": True,
        "lr": 2e-05, "warmup": 0.1,
    },
    "E7_focal_loss": {
        "preproc": "raw", "context": "none", "weighted": False,
        "lr": 1e-5, "warmup": 0.0, "loss": "focal", "gamma": 2.0,
    },
}
assert all(r in EXPERIMENTS for r in RUN_NAMES)


## 5. Metrik, tokenisasi, dan penyimpanan JSON

In [11]:
def compute_metrics(prediction):
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    y_true = prediction.label_ids
    y_pred = np.asarray(logits).argmax(axis=-1)
    report = classification_report(
        y_true, y_pred, labels=[0, 1, 2], target_names=LABELS, output_dict=True, zero_division=0
    )
    scores = {
        "macro_f1": float(f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=[0, 1, 2], average="weighted", zero_division=0)),
    }
    for label in LABELS:
        for metric in ("f1-score", "recall", "precision"):
            name = "f1" if metric == "f1-score" else metric
            scores[f"{name}_{label}"] = float(report[label][metric])
    return scores


def tokenize_split(frame, tokenizer, conf):
    texts = frame["yt_comment"].copy()
    if conf["preproc"] == "legacy":
        texts = texts.map(lambda text: emoji.replace_emoji(text, replace=" [EMOJI] "))
    if conf["preproc"] in ("legacy", "keep_emoji"):
        texts = texts.str.casefold()

    context = None
    if conf["context"] == "title":
        context = frame["yt_title"].tolist()
    elif conf["context"] == "title_desc":
        context = (frame["yt_title"].str.strip() + ". " + frame["yt_description"].str[:200]).tolist()

    tokens = tokenizer(
        texts.tolist(),
        text_pair=context, max_length=MAX_LENGTH,
        truncation="longest_first" if context is not None else True,
    )
    tokens["labels"] = frame["hate_speech"].map(LABEL2ID).tolist()
    return Dataset.from_dict(tokens)


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    value = json.loads(json.dumps(value, ensure_ascii=False), parse_constant=str)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def compute_loss(outputs, labels, num_items_in_batch=None, class_weights=None, gamma=0.0):
    logits = outputs.logits.float()
    weights = class_weights.to(logits.device) if class_weights is not None else None
    if gamma == 0:
        return torch.nn.functional.cross_entropy(logits, labels, weight=weights)

    # Hitung probabilitas kelas benar dari CE tanpa class weights.
    ce = torch.nn.functional.cross_entropy(logits, labels, reduction="none")
    loss = (1 - torch.exp(-ce)).pow(gamma) * ce
    if weights is not None:
        loss = loss * weights[labels]
    return loss.mean()


## 6. Training dan evaluasi validation

In [12]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA tidak tersedia.")

records = []
for lang in LANGUAGES:
    for model_name in MODEL_NAMES:
        for run_name in RUN_NAMES:
            print(f"\n{lang.upper()} | {model_name} | {run_name}")
            conf = EXPERIMENTS[run_name]
            directory = OUT_DIR / lang / model_name.split("/")[-1] / run_name
            metrics_file = directory / "validation_metrics.json"
            if SKIP_FINISHED and metrics_file.exists():
                print(f"Melewati {lang}/{model_name}/{run_name}: sudah selesai sebelumnya")
                records.append(json.loads(metrics_file.read_text(encoding="utf-8")))
                continue
            directory.mkdir(parents=True, exist_ok=True)
            set_seed(SEED)
            cleanup()
            trainer = model = train_ds = val_ds = tokenizer = None
            pred = logits = y_true = y_pred = validation_pred = None
            try:
                tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
                train_ds = tokenize_split(DATA[lang]["train"], tokenizer, conf)
                val_ds = tokenize_split(DATA[lang]["val"], tokenizer, conf)
                class_counts = (
                    DATA[lang]["train"]["hate_speech"]
                    .map(LABEL2ID)
                    .value_counts()
                    .reindex(range(3))
                    .to_numpy()
                )
                class_weights = torch.tensor(
                    len(DATA[lang]["train"]) / (3 * class_counts), dtype=torch.float32
                )

                model = AutoModelForSequenceClassification.from_pretrained(
                    model_name, num_labels=3, label2id=LABEL2ID, id2label=ID2LABEL
                )
                model.config.use_cache = False
                args = TrainingArguments(
                    output_dir=str(directory / "checkpoints"), eval_strategy="epoch",
                    save_strategy="best", logging_strategy="epoch",
                    learning_rate=conf["lr"], warmup_ratio=conf["warmup"],
                    weight_decay=WEIGHT_DECAY, num_train_epochs=NUM_EPOCHS,
                    per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
                    gradient_accumulation_steps=1, seed=SEED,
                    data_seed=SEED, gradient_checkpointing=GRADIENT_CHECKPOINTING,
                    eval_accumulation_steps=EVAL_ACCUMULATION_STEPS, fp16=torch.cuda.is_available(),
                    bf16=False, load_best_model_at_end=True,
                    metric_for_best_model="macro_f1", greater_is_better=True,
                    save_total_limit=1, save_safetensors=True, save_only_model=True,
                    report_to="none", dataloader_num_workers=0,
                    optim="adamw_torch", lr_scheduler_type="linear",
                )
                loss_weights = class_weights if conf["weighted"] else None
                trainer = Trainer(
                    model=model, args=args,
                    train_dataset=train_ds, eval_dataset=val_ds,
                    processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                    compute_metrics=compute_metrics,
                    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
                    compute_loss_func=partial(
                        compute_loss, class_weights=loss_weights, gamma=conf.get("gamma", 0.0),
                    ),
                )
                trainer.model_accepts_loss_kwargs = False
                trainer.train()
                pred = trainer.predict(val_ds)
                logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
                y_true = pred.label_ids
                y_pred = np.argmax(logits, axis=-1)
                metrics = compute_metrics(pred)
                metrics.update(
                    {
                        "language": lang,
                        "model": model_name,
                        "run_name": run_name,
                        "best_checkpoint": str(trainer.state.best_model_checkpoint),
                        "seed": SEED,
                        "data_sha256": hashlib.sha256(data_path(lang, "train").read_bytes()).hexdigest(),
                    }
                )
                save_json(directory / "configuration.json", conf)
                pd.DataFrame(trainer.state.log_history).to_csv(
                    directory / "training_history.csv", index=False
                )
                pd.DataFrame(
                    classification_report(
                        y_true,
                        y_pred,
                        labels=[0, 1, 2], target_names=LABELS,
                        output_dict=True, zero_division=0,
                    )
                ).transpose().to_csv(directory / "classification_report_validation.csv")
                pd.DataFrame(
                    confusion_matrix(y_true, y_pred, labels=[0, 1, 2]), index=LABELS, columns=LABELS
                ).to_csv(directory / "confusion_matrix_validation.csv", index_label="actual_label")
                validation_pred = DATA[lang]["val"][
                    ["StereoQueerEval_id", "yt_comment", "yt_title", "hate_speech"]
                ].copy()
                validation_pred["predicted_label"] = [ID2LABEL[int(v)] for v in y_pred]
                validation_pred["correct"] = (
                    validation_pred["hate_speech"] == validation_pred["predicted_label"]
                )
                validation_pred.to_csv(directory / "predictions_validation.csv", index=False)
                # Tandai lengkap PALING AKHIR agar kegagalan menyimpan tidak dianggap selesai.
                save_json(metrics_file, metrics)
                # Simpan satu pemenang untuk setiap pasangan model dan bahasa.
                model_key = model_name.split("/")[-1] + "/" + lang
                previous = best_models.get(model_key)
                if previous is None or metrics["macro_f1"] > previous["score"]:
                    best_models[model_key] = {
                        "model": model_name.split("/")[-1], "language": lang,
                        "score": metrics["macro_f1"], "path": metrics["best_checkpoint"],
                        "run": str(directory.resolve()),
                    }
                    save_json(BEST_MODELS_FILE, best_models)

                winner_path = Path(best_models[model_key]["path"]).resolve()
                obsolete = list((directory / "checkpoints").glob("checkpoint-*"))
                if previous is not None:
                    obsolete.append(Path(previous["path"]))
                for checkpoint in obsolete:
                    checkpoint = checkpoint.resolve()
                    if checkpoint != winner_path and checkpoint.exists():
                        assert checkpoint.is_relative_to(OUT_DIR.parent.resolve())
                        assert checkpoint.name.startswith("checkpoint-") or checkpoint.name == "best_model"
                        shutil.rmtree(checkpoint)
                records.append(metrics)
                comparison = pd.DataFrame(records)
                comparison.to_csv(OUT_DIR / "SEMUA_HASIL_VALIDASI.csv", index=False)
                print(f"Validation macro-F1: {metrics['macro_f1']:.4f}")
            finally:
                # Lepaskan referensi sebelum membersihkan cache GPU.
                del trainer, model, train_ds, val_ds, tokenizer
                del pred, logits, y_true, y_pred, validation_pred
                cleanup()



EN | cardiffnlp/twitter-xlm-roberta-base | E0_legacy
Melewati en/cardiffnlp/twitter-xlm-roberta-base/E0_legacy: sudah selesai sebelumnya

EN | cardiffnlp/twitter-xlm-roberta-base | E1_keep_emoji
Melewati en/cardiffnlp/twitter-xlm-roberta-base/E1_keep_emoji: sudah selesai sebelumnya

EN | cardiffnlp/twitter-xlm-roberta-base | E2_keep_case
Melewati en/cardiffnlp/twitter-xlm-roberta-base/E2_keep_case: sudah selesai sebelumnya

EN | cardiffnlp/twitter-xlm-roberta-base | E3_no_weights
Melewati en/cardiffnlp/twitter-xlm-roberta-base/E3_no_weights: sudah selesai sebelumnya

EN | cardiffnlp/twitter-xlm-roberta-base | E4_title_pair
Melewati en/cardiffnlp/twitter-xlm-roberta-base/E4_title_pair: sudah selesai sebelumnya

EN | cardiffnlp/twitter-xlm-roberta-base | E5_title_desc_pair
Melewati en/cardiffnlp/twitter-xlm-roberta-base/E5_title_desc_pair: sudah selesai sebelumnya

EN | cardiffnlp/twitter-xlm-roberta-base | E6_lr_warmup
Melewati en/cardiffnlp/twitter-xlm-roberta-base/E6_lr_warmup: sudah

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyboardInterrupt: 

## 7. Perbandingan hasil validation

In [ ]:
records = []
for lang in LANGUAGES:
    for model_name in MODEL_NAMES:
        for run_name in RUN_NAMES:
            path = OUT_DIR / lang / model_name.split("/")[-1] / run_name / "validation_metrics.json"
            if path.exists():
                records.append(json.loads(path.read_text(encoding="utf-8")))
comparison = pd.DataFrame(records)
if comparison.empty:
    raise RuntimeError("Belum ada hasil validation.")
comparison = comparison.sort_values(["language", "model", "macro_f1"], ascending=[True, True, False])
comparison.to_csv(OUT_DIR / "SEMUA_HASIL_VALIDASI.csv", index=False)
cols = [
    "language",
    "model",
    "run_name",
    "macro_f1",
    "accuracy",
    "f1_no",
    "f1_yes_implicit",
    "f1_yes_explicit",
]
display(
    comparison[cols].style.format({x: "{:.4f}" for x in cols if x not in ("language", "model", "run_name")})
)

winners = comparison.sort_values("macro_f1", ascending=False).drop_duplicates(["language", "model"])
print("PEMENANG VALIDASI UNTUK SETIAP PASANG MODEL/BAHASA:")
display(winners[cols])
for (lang, model_name), group in comparison.groupby(["language", "model"], sort=False):
    order = [x for x in RUN_NAMES if x in set(group["run_name"])]
    group = group.set_index("run_name").loc[order]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(group.index, group["macro_f1"])
    ax.set_ylim(0, 1)
    ax.set(ylabel="Macro F1 (validation)", title=f"{lang.upper()} | {model_name}")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    out = OUT_DIR / lang / model_name.split("/")[-1] / "comparison_macro_f1.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=160)
    plt.show()
    plt.close(fig)

print("Model yang dipertahankan tercatat di:", BEST_MODELS_FILE)
print("Checkpoint eksperimen kalah sudah dibersihkan setelah masing-masing run.")


## 8. Evaluasi test konfigurasi terpilih

In [ ]:
if not RUN_FINAL_TEST:
    print("Evaluasi test dikunci. Tinjau hasil validasi dahulu; ubah RUN_FINAL_TEST=True setelah final.")
else:
    assert not comparison.empty, "Hasil validasi belum tersedia"
    # Jika memilih lintas model, pastikan FINAL_MODEL dan FINAL_RUN ditetapkan berdasarkan validasi saja.
    for FINAL_LANGUAGE in LANGUAGES:
        for FINAL_MODEL in MODEL_NAMES:
            final_pool = comparison[
                (comparison["language"] == FINAL_LANGUAGE) & (comparison["model"] == FINAL_MODEL)
            ]
            assert len(final_pool) == len(RUN_NAMES), "Selesaikan seluruh eksperimen sebelum tes."
            FINAL_RUN = final_pool.sort_values("macro_f1", ascending=False).iloc[0]["run_name"]
            print("Final (berdasarkan validasi):", FINAL_LANGUAGE, FINAL_MODEL, FINAL_RUN)
            destination = OUT_DIR / FINAL_LANGUAGE / FINAL_MODEL.split("/")[-1] / FINAL_RUN
            target = destination / "test_metrics.json"
            if target.exists():
                print("Hasil test sudah ada, tidak dihitung ulang:")
                print(target.read_text(encoding="utf-8"))
            else:
                checkpoint = final_pool[final_pool["run_name"] == FINAL_RUN].iloc[0]["best_checkpoint"]
                if not Path(checkpoint).exists():
                    kept = best_models.get(FINAL_MODEL.split("/")[-1] + "/" + FINAL_LANGUAGE)
                    if kept and Path(kept["path"]).exists():
                        print("Pemenang tersimpan berasal dari run lain:", kept["path"])
                        print("Gunakan notebook dan preprocessing run tersebut untuk evaluasi test.")
                        continue
                    raise FileNotFoundError(f"Checkpoint tidak ditemukan: {checkpoint}")
                cleanup()
                trainer = model = test_ds = tokenizer = pred = logits = y_true = y_pred = None
                try:
                    tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL, use_fast=True)
                    model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
                    conf = EXPERIMENTS[FINAL_RUN]
                    test_ds = tokenize_split(DATA[FINAL_LANGUAGE]["test"], tokenizer, conf)
                    args = TrainingArguments(
                        output_dir=str(destination / "test_only"), per_device_eval_batch_size=EVAL_BATCH_SIZE,
                        eval_accumulation_steps=EVAL_ACCUMULATION_STEPS, report_to="none",
                    )
                    trainer = Trainer(
                        model=model, args=args,
                        processing_class=tokenizer,
                        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                        compute_metrics=compute_metrics,
                    )
                    pred = trainer.predict(test_ds)
                    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
                    y_true = pred.label_ids
                    y_pred = np.asarray(logits).argmax(axis=-1)
                    final_scores = compute_metrics(pred)
                    pd.DataFrame(
                        confusion_matrix(y_true, y_pred, labels=[0, 1, 2]), index=LABELS, columns=LABELS
                    ).to_csv(destination / "confusion_matrix_test.csv", index_label="actual_label")
                    pd.DataFrame(
                        classification_report(
                            y_true,
                            y_pred,
                            labels=[0, 1, 2], target_names=LABELS,
                            output_dict=True, zero_division=0,
                        )
                    ).transpose().to_csv(destination / "classification_report_test.csv")
                    save_json(
                        target,
                        {
                            "language": FINAL_LANGUAGE,
                            "model": FINAL_MODEL,
                            "run_name": FINAL_RUN,
                            "scores": final_scores,
                        },
                    )
                    print("TEST INTERNAL (bukan official test SemEval):")
                    display(pd.DataFrame([final_scores]))
                    print(
                        classification_report(
                            y_true, y_pred, labels=[0, 1, 2], target_names=LABELS, zero_division=0
                        )
                    )
                finally:
                    del trainer, model, test_ds, tokenizer, pred, logits, y_true, y_pred
                    cleanup()


## 9. Ekspor laporan ZIP

In [ ]:
if comparison.empty:
    print("Belum ada hasil eksperimen yang dapat diekspor.")
else:
    archive_base = OUT_DIR / "laporan_ablation_e0_e7"
    with ZipFile(str(archive_base) + ".zip", "w", compression=ZIP_DEFLATED) as archive:
        for file in OUT_DIR.rglob("*"):
            if (
                file.is_file()
                and "checkpoints" not in file.parts
                and file.suffix.lower() != ".zip"
            ):
                archive.write(file, arcname=file.relative_to(OUT_DIR))
    print("CSV:", OUT_DIR / "SEMUA_HASIL_VALIDASI.csv")
    print("ZIP:", str(archive_base) + ".zip")
    display(FileLink(str(archive_base) + ".zip"))
    print("Catatan: kirim CSV hasil ke chat agar saya bedah perbedaan semua delapan eksperimen.")
